# 1 - Importação das Bibliotecas

O notebook irá realizar uso da base de dados que está localizado no google drive.

Também será utilizadas bibliotecas do numpy, pandas e matplotlib para manipulação dos dados e visualização.

O notebook tdqm será utilizado para construção da barra de progresso.

O tensorflow será utilizado para construção e utilização do modelo MLP.

Já o sklearn será utilizado para os modelos MLP, RandomForest.

Por fim, o xgboost será utilizado para implementação do modelo XGBoost.

In [ ]:
from google.colab import drive

import os
import gc
from datetime import datetime
import time

import numpy as np
import pandas as pd
from math import trunc


import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

import shap

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2



In [ ]:
# Checagem de disponibilidade da GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ {len(gpus)} GPU(s) disponível(is): {[gpu.name for gpu in gpus]}")
else:
    print("⚠️ Nenhuma GPU disponível. Usando CPU.")

# Função de Criação de Valores SHAP

In [ ]:
def cria_valores_shap(df, modelo_nome, nome_grupo, threshold, n_splits=5,
                      n_background_kmeans=10, n_test_explicar=1000,
                      nsamples=1000, l1_num_features=500,
                      batch_size_teste=10, predict_batch_size=8192,
                      checkpoint_a_cada=50, usar_tf_function=True):


    X = df.drop(columns=['classe']).values.astype(np.int8)
    y = df['classe'].astype(np.int8).values

    print(f"\nIniciando avaliação para o grupo de features: '{nome_grupo}' "
          f"com {X.shape[1]} atributos e {X.shape[0]} instâncias.")
    print(f"Config: n_bg_kmeans={n_background_kmeans}, "
          f"n_test_explicar={n_test_explicar}, nsamples={nsamples}, "
          f"l1_num_features={l1_num_features}, "
          f"predict_batch_size={predict_batch_size}, "
          f"usar_tf_function={usar_tf_function}")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    folds = list(skf.split(X, y))

    CAMINHO_SAIDA_BASE = (
        f'DIRETORIO_BASE/src_shap/{nome_grupo}/'
        f'{modelo_nome}_limiar{trunc(threshold*10)}/'
    )
    os.makedirs(CAMINHO_SAIDA_BASE, exist_ok=True)

    caminho_log = os.path.join(
        CAMINHO_SAIDA_BASE,
        f"log_tempo_shap_{nome_grupo}_{modelo_nome}_limiar{trunc(threshold*10)}.csv"
    )

    registros_log = []

    for fold in range(n_splits):

        tempo_fold_inicio = time.perf_counter()

        train_idx, test_idx = folds[fold]
        feature_names = np.array([c for c in df.columns if c != 'classe'])

        print(f"\nFold {fold+1}/{n_splits}")
        print(f"Treino: {train_idx.shape[0]} | Teste: {test_idx.shape[0]}")

        # --- Subset estratificado do teste ---
        if n_test_explicar is None:
            test_idx_explicar = test_idx
        else:
            y_test_full = y[test_idx]
            idx_pos = test_idx[y_test_full == 1]
            idx_neg = test_idx[y_test_full == 0]
            rng_fold = np.random.default_rng(42 + fold)
            n_por_classe = n_test_explicar // 2
            sel_pos = rng_fold.choice(idx_pos,
                                      size=min(n_por_classe, len(idx_pos)),
                                      replace=False)
            sel_neg = rng_fold.choice(idx_neg,
                                      size=min(n_test_explicar - len(sel_pos),
                                               len(idx_neg)),
                                      replace=False)
            test_idx_explicar = np.concatenate([sel_pos, sel_neg])
            test_idx_explicar = test_idx_explicar[
                rng_fold.permutation(len(test_idx_explicar))
            ]
            del y_test_full, idx_pos, idx_neg

        X_test = X[test_idx_explicar].astype(np.float32)
        y_test = y[test_idx_explicar].astype(np.int32)
        index_explicado = test_idx_explicar.astype(np.int64)
        n_test = X_test.shape[0]

        print(f"Teste a explicar (estratificado): {n_test} amostras "
              f"({(y_test == 1).sum()} pos, {(y_test == 0).sum()} neg)")

        # --- Carregamento do modelo ---
        print("Carregando o modelo MLP Keras.")
        CAMINHO_PASTA_MODELO = (
            f'DIRETORIO_BASE/src_inicial/{nome_grupo}/'
            f'{modelo_nome}_limiar{trunc(threshold*10)}/resultados/'
        )
        CAMINHO_MODELO = os.path.join(
            CAMINHO_PASTA_MODELO,
            f'{nome_grupo}__{modelo_nome}__fold{fold+1}.keras'
        )

        tempo_modelo_inicio = time.perf_counter()
        modelo = tf.keras.models.load_model(CAMINHO_MODELO)
        tempo_modelo_fim = time.perf_counter()

        # Probe inicial para descobrir formato da saída
        _probe = modelo.predict(X_test[0:1], batch_size=1, verbose=0)
        saida_eh_escalar = (_probe.ndim == 2 and _probe.shape[1] == 1)
        n_classes_saida = 1 if saida_eh_escalar else _probe.shape[1]
        del _probe

        # --- Função de predição otimizada ---
        if usar_tf_function:
            @tf.function(reduce_retracing=True)
            def _predict_compiled(x_tensor):
                return modelo(x_tensor, training=False)

            def f_modelo(x):
                x = np.asarray(x)
                if x.ndim == 1:
                    x = x.reshape(1, -1)
                if x.shape[0] == 0:
                    if saida_eh_escalar:
                        return np.zeros((0,), dtype=np.float32)
                    return np.zeros((0, n_classes_saida), dtype=np.float32)

                n = x.shape[0]
                x_f32 = x.astype(np.float32, copy=False)

                # Batching manual para controlar pico de VRAM
                if n <= predict_batch_size:
                    preds = _predict_compiled(tf.constant(x_f32)).numpy()
                else:
                    out_parts = []
                    for i in range(0, n, predict_batch_size):
                        chunk = x_f32[i:i + predict_batch_size]
                        out_parts.append(
                            _predict_compiled(tf.constant(chunk)).numpy()
                        )
                    preds = np.concatenate(out_parts, axis=0)

                if saida_eh_escalar:
                    return preds.ravel()
                return preds
        else:
            # Fallback via model.predict
            def f_modelo(x):
                x = np.asarray(x)
                if x.ndim == 1:
                    x = x.reshape(1, -1)
                if x.shape[0] == 0:
                    if saida_eh_escalar:
                        return np.zeros((0,), dtype=np.float32)
                    return np.zeros((0, n_classes_saida), dtype=np.float32)
                preds = modelo.predict(x.astype(np.float32),
                                       batch_size=predict_batch_size,
                                       verbose=0)
                if saida_eh_escalar:
                    return preds.ravel()
                return preds

        # --- Background via kmeans ---
        rng_bg = np.random.default_rng(42 + fold)
        bg_pool_idx = rng_bg.choice(train_idx, size=min(500, len(train_idx)),
                                    replace=False)
        bg_pool = X[bg_pool_idx].astype(np.float32)
        background = shap.kmeans(bg_pool, n_background_kmeans)
        del bg_pool

        # --- Criação do explainer ---
        tempo_explainer_inicio = time.perf_counter()
        explainer = shap.KernelExplainer(f_modelo, background)
        tempo_explainer_fim = time.perf_counter()

        # Warm-up do tf.function — primeira chamada compila o grafo (lenta).
        # Fazendo aqui evita que o primeiro batch real pague esse custo.
        if usar_tf_function:
            print("Warm-up do tf.function...")
            t_warm = time.perf_counter()
            _ = f_modelo(X_test[0:1])
            _ = f_modelo(X_test[0:min(predict_batch_size, n_test)])
            print(f"Warm-up concluído em {time.perf_counter() - t_warm:.1f}s.")

        base_value_scalar = float(explainer.expected_value
                                  if np.isscalar(explainer.expected_value)
                                  else explainer.expected_value[0])

        # --- Arquivo NPZ final + memmap incremental ---
        saida_npz = os.path.join(
            CAMINHO_SAIDA_BASE,
            f"{nome_grupo}__{modelo_nome}__fold{fold+1}_SHAP.npz"
        )
        saida_memmap = saida_npz.replace('.npz', '_shap.memmap')
        shap_mat = np.memmap(saida_memmap, dtype=np.float32, mode='w+',
                             shape=(n_test, X.shape[1]))

        # --- Loop principal ---
        tempo_shap_inicio = time.perf_counter()
        l1_reg_arg = f'num_features({l1_num_features})'

        for ini in range(0, n_test, batch_size_teste):
            fim = min(ini + batch_size_teste, n_test)
            batch_inicio = time.perf_counter()

            sv = explainer.shap_values(X_test[ini:fim],
                                       nsamples=nsamples,
                                       l1_reg=l1_reg_arg,
                                       silent=True)

            if isinstance(sv, list):
                sv_arr = sv[1] if len(sv) > 1 else sv[0]
            else:
                sv_arr = sv
            sv_arr = np.asarray(sv_arr, dtype=np.float32)
            if sv_arr.ndim == 1:
                sv_arr = sv_arr.reshape(1, -1)

            shap_mat[ini:fim, :] = sv_arr
            batch_fim = time.perf_counter()

            if (fim % checkpoint_a_cada == 0) or (fim == n_test):
                shap_mat.flush()

            elapsed = batch_fim - tempo_shap_inicio
            taxa = fim / max(elapsed, 1e-6)
            restante = (n_test - fim) / max(taxa, 1e-6)
            print(f"  [{fim}/{n_test}] batch={fim-ini}: {batch_fim-batch_inicio:.1f}s "
                  f"| acumulado: {elapsed/60:.1f}min "
                  f"| ETA: {restante/60:.1f}min")

            del sv, sv_arr
            gc.collect()

        tempo_shap_fim = time.perf_counter()
        shap_mat.flush()

        # --- Consolidação final do NPZ ---
        tempo_salvamento_inicio = time.perf_counter()
        base_values = np.full(n_test, base_value_scalar, dtype=np.float32)

        np.savez_compressed(
            saida_npz,
            shap_values=np.asarray(shap_mat),
            base_values=base_values,
            feature_names=feature_names,
            X_explicado=X_test,
            test_index=index_explicado,
            y_test=y_test
        )

        del shap_mat
        gc.collect()
        try:
            os.remove(saida_memmap)
        except Exception:
            pass

        tempo_salvamento_fim = time.perf_counter()
        tempo_fold_fim = time.perf_counter()

        registro = {
            "data_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "nome_grupo": nome_grupo,
            "modelo_nome": modelo_nome,
            "threshold": threshold,
            "fold": fold + 1,
            "explainer": "KernelExplainer",
            "usar_tf_function": usar_tf_function,
            "n_features": X.shape[1],
            "n_train": len(train_idx),
            "n_test_total": len(test_idx),
            "n_test_explicado": n_test,
            "n_background_kmeans": n_background_kmeans,
            "nsamples": nsamples,
            "l1_num_features": l1_num_features,
            "batch_size_teste": batch_size_teste,
            "predict_batch_size": predict_batch_size,
            "tempo_carregamento_modelo_seg": tempo_modelo_fim - tempo_modelo_inicio,
            "tempo_criacao_explainer_seg": tempo_explainer_fim - tempo_explainer_inicio,
            "tempo_calculo_shap_seg": tempo_shap_fim - tempo_shap_inicio,
            "tempo_por_amostra_seg": (tempo_shap_fim - tempo_shap_inicio) / max(n_test, 1),
            "tempo_salvamento_seg": tempo_salvamento_fim - tempo_salvamento_inicio,
            "tempo_total_fold_seg": tempo_fold_fim - tempo_fold_inicio,
            "caminho_modelo": CAMINHO_MODELO,
            "caminho_saida_npz": saida_npz
        }

        registros_log.append(registro)
        pd.DataFrame(registros_log).to_csv(caminho_log, index=False)

        print(f"\nFold {fold+1} concluído:")
        print(f"  Tempo SHAP: {registro['tempo_calculo_shap_seg']:.1f}s "
              f"({registro['tempo_por_amostra_seg']:.2f}s/amostra)")
        print(f"  Tempo total: {registro['tempo_total_fold_seg']:.1f}s")
        print(f"  Salvo em: {saida_npz}")

        del explainer, modelo, background, f_modelo
        del X_test, y_test, index_explicado, base_values
        if usar_tf_function:
            del _predict_compiled
        tf.keras.backend.clear_session()
        gc.collect()

# 2 - Abertura do Arquivo, recuperação dos dados e embaralhamento

Nessa etapa é realizada o acesso ao drive e ao arquivo.

O arquivo referenciado é a base que contém 79798 amostras de benignos e 79798 de malwares, sem duplicação.

Para compreender melhor sobre o conjunto de dados, são impressos na tela os formatos de X e y, a quantidade de benignos e de malwares.

Como a base de dados foi construída com a eliminação de duplicados de malwares e com a mesma porção de benignos. Essas amostras no final foram concatenadas e ficando agrupadas, portanto, **necessário realizar o embaralhamento das amostras**.

In [ ]:
# Acesso ao drive pessoal
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Caminho para o arquivo compactado
CAMINHO_ARQUIVO = 'DIRETORIO_BASE/dados/mh1m_balanceadas.npz'

# Carrega os dados com mmap_mode para uso mais leve de memória
dados = np.load(CAMINHO_ARQUIVO, allow_pickle=True)

# Extração dos arrays principais
X = dados['data']
y = dados['classes']
colunas = dados['column_names']

In [ ]:
# Embaralhar X e y
rng = np.random.default_rng(42)  # garante reprodutibilidade
idx_final = rng.permutation(X.shape[0])  # embaralha os índices

X = X[idx_final]
y = y[idx_final]

print(f"Dados embaralhados: X={X.shape}, y={y.shape}")

Dados embaralhados: X=(159020, 23239), y=(159020,)


# 3 - Separar as colunas das features e criar os **DataFrames** e **Execucao**

São separados os índices de cada grupo de características, sendo separadas pelos namespaces.

Para cada grupo são criados dataframes para serem manipulados em momentos separados.

Ao final, são impressos os formatos dos dataframes de cada um,

Além dos namespaces definidos, foram criados dois grupos mais sendo um para a união de permissions e opcodes e outro com a junção de todas as características.


In [ ]:
modelo_nome = "mlp"
threshold = 0.5
grupos = ["intents", "permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]


for nome_grupo in grupos:

  # Identificar colunas por namespace

  if nome_grupo == "permissions_opcodes":
      idx_permissions = [i for i, nome in enumerate(colunas) if nome.startswith("permissions::")]
      idx_opcodes = [i for i, nome in enumerate(colunas) if nome.startswith("opcodes::")]
      idx_features = idx_permissions + idx_opcodes
  elif nome_grupo == "todas":
      idx_features = range(len(colunas))
  else:
      idx_features = [i for i, nome in enumerate(colunas) if nome.startswith(f"{nome_grupo}::")]


  # Criação dos DataFrames
  df    = pd.DataFrame(X[:, idx_features], columns=np.array(colunas)[idx_features])
  df['classe'] = y

  print("DataFrames criados:")

  print(f" - df: {df.shape}")

  cria_valores_shap(df, modelo_nome, nome_grupo, threshold, 5)
  del df
  gc.collect()
